In [19]:
from datetime import date,timedelta
import httpx

today = date.today() - timedelta(days=1)
data = {
    "query": f"publication-date={today.strftime('%Y%m%d')}",
    "fields": [
        "publication-date",
        "notice-title",
        "procedure-type",
        "contract-nature",
        # TODO fix country
        # "country",
        "tender-value",
        "tender-value-cur",
        "classification-cpv",
        "organisation-contact-point-tenderer",
        "document-url-lot"
    ],
    "page": 1,
    "limit": 10,
    "scope": "ACTIVE",
    "checkQuerySyntax": False,
    "paginationMode": "ITERATION"
}

r = httpx.post('https://api.ted.europa.eu/v3/notices/search', json=data)
r.json()

{'notices': [{'organisation-contact-point-tenderer': ['Jiri Stanek',
    'Johan Håkansson'],
   'procedure-type': 'open',
   'classification-cpv': ['77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77300000',
    '77300000',
    '77300000',
    '77300000',
    '77300000',
    '77300000'],
   'publication-number': '32737-2025',
   'contract-nature': ['services',
    'services',
    'services',
    'services',
    'services',
    'services'],
   'publication-date': '2025-01-17+01:00',
   'links': {'xml': {'MUL': 'https://ted.europa.eu/en/notice/32737-2025/xml'},
    'pdf': {'BUL': 'https://ted.europa.eu/bg/notice/32737-2025/pdf',
     'SPA': 'https://ted.europa.eu/es/notice/32737-2025/pdf',
     'CES': 'https://ted.europa.eu/cs/notice/32737-2025/pdf',
 

In [1]:
import os

import httpx
from dotenv import load_dotenv
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session

load_dotenv(".env")


client_id = os.environ["PUBPROC_CLIENT_ID"]
client_secret = os.environ["PUBPROC_CLIENT_SECRET"]

url = "https://public.pr.fedservices.be/api/oauth2/token"

client = BackendApplicationClient(client_id=client_id)
oauth = OAuth2Session(client=client)

token = oauth.fetch_token(
    token_url=url,
    client_id=client_id,
    client_secret=client_secret,
    include_client_id=True,
)

print(token["access_token"], token["expires_at"])

5FhBZK3NztGATVzjSIMdj2rykMGachkhUSOib1CyrwUvS6ve2ITB1T 1743731864.042791


/Users/mehmet/projects/proclogic_api/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [1]:
from datetime import date, timedelta

def get_nearest_business_day(date_obj: date = None) -> date:
    if date_obj is None:
        date_obj = date.today()

    if date_obj.weekday() == 5:  # Saturday
        return date_obj - timedelta(days=1)
    elif date_obj.weekday() == 6:  # Sunday
        return date_obj - timedelta(days=2)
    return date_obj

latest_business_day = get_nearest_business_day()

data = {
    "dispatch-date-from": "2025-01-01",
    "page": 1,
    "pageSize": 100,
}

headers = {
    "Authorization": f"Bearer {token['access_token']}",
    "BelGov-Trace-Id": "2ce83af9-d524-43a6-8d1c-b19dff051aed",
}
r = httpx.get('https://public.pr.fedservices.be/api/eProcurementSea/v1/search/publications', params=data, headers=headers)

publications = r.json()["publications"]
print("total count", r.json()["totalCount"])

if r.status_code == 200:
    totalCount = int(r.json()["totalCount"])
    pages = totalCount / 100 if totalCount % 100 == 0 else int(totalCount / 100) + 1

    if pages > 1:
        for i in range(2, pages + 1):
            data["page"] = i
            r = httpx.get(
                "https://public.pr.fedservices.be/api/eProcurementSea/v1/search/publications",
                params=data,
                headers=headers,
            )
            publications.extend(r.json()["publications"])
            break
        
publications

NameError: name 'token' is not defined

In [2]:

headers = {
    "Authorization": f"Bearer {token['access_token']}",
    "BelGov-Trace-Id": "2ce83af9-d524-43a6-8d1c-b19dff051aed",
}

In [10]:
import asyncio
import logging
import os
import re
import shutil
import struct
import subprocess
import tempfile
import uuid
from io import BytesIO
from typing import Dict, List

import httpx
import zipfile
from app.config.settings import settings
from app.util.pubproc_token import get_token
from app.util.redis_cache import redis_cache




def generate_uuid():
    return str(uuid.uuid4())

def try_fix_zip(zip_bytes: bytes) -> BytesIO:
    """
    Attempt to fix common ZIP corruption issues:
    - Remove trailing nulls or junk bytes after EOCD
    """
    # Find EOCD (End of Central Directory) signature
    eocd_signature = b"\x50\x4b\x05\x06"  # PK\x05\x06

    eocd_index = zip_bytes.rfind(eocd_signature)
    if eocd_index == -1:
        raise zipfile.BadZipFile("EOCD not found in zip file.")

    # Truncate to EOCD + 22 bytes (minimum EOCD size)
    fixed_bytes = zip_bytes[:eocd_index + 22]

    return BytesIO(fixed_bytes)

def get_publication_workspace_documents(
    publication_workspace_id: str
) -> dict:
    """
    Get document files for a publication workspace with robust ZIP handling.
    
    Args:
        client: HTTP client
        publication_workspace_id: ID of the publication workspace
        
    Returns:
        Dictionary mapping filenames to BytesIO objects
    """
    token = get_token()
    headers = {
        "Authorization": f"Bearer {token}",
        "BelGov-Trace-Id": generate_uuid(),
    }

    temp_dir = None
    extract_dir = None
    temp_file_path = None
    
    
    # Download the archive
    print(f"Downloading archive for {publication_workspace_id} to {temp_file_path}")
    
    # Add a 5-minute timeout for large files
    r = httpx.get(
        "https://public.pr.fedservices.be"
        + "/api/eProcurementDos/v1"
        + f"/publication-workspaces/7ae51ab1-56ed-45db-b5b8-3683bf8724b5/archive", #fd93143b-f799-4c81-9a88-475ad86a400d 3 aprillll
        # TODO: /publication-workspaces/{publication-workspace-id}/urls
        headers=headers,
        timeout=300,  # 5 minutes
    )

    zip_bytes = r.content

    # Debug save to disk
    with open("debug_downloaded.zip", "wb") as f:
        f.write(zip_bytes)

    try:
        primary_zip = zipfile.ZipFile(BytesIO(zip_bytes))
    except zipfile.BadZipFile:
        print("Initial zip file corrupt, attempting repair...")
        primary_zip = zipfile.ZipFile(try_fix_zip(zip_bytes))


    if r.status_code != 200:
        logging.warning(f"Failed to fetch archive for {publication_workspace_id}: {r.status_code}")
        return {}
    
    file_map = {}
    # First level zip extraction
    for file_name in primary_zip.namelist():
        file_content = primary_zip.read(file_name)

        # Get just the base filename without folder path
        base_file_name = os.path.basename(file_name)

        # Skip if it's a directory (empty base name)
        if not base_file_name:
            continue

        # Check if this is another zip file (second level)
        if base_file_name.lower().endswith(".zip"):
            try:
                with zipfile.ZipFile(BytesIO(file_content)) as secondary_zip:
                    for inner_file_name in secondary_zip.namelist():
                        # Get just the base filename for inner files too
                        inner_base_name = os.path.basename(inner_file_name)

                        # Skip if it's a directory
                        if not inner_base_name:
                            continue

                        inner_content = secondary_zip.read(inner_file_name)
                        inner_file = BytesIO(inner_content)
                        inner_file.name = inner_base_name
                        file_map[inner_base_name] = inner_file
            except zipfile.BadZipFile:
                # If it's not actually a valid zip, treat it as a regular file
                file_data = BytesIO(file_content)
                file_data.name = base_file_name
                file_map[base_file_name] = file_data
        else:
            # Regular file, not a zip
            file_data = BytesIO(file_content)
            file_data.name = base_file_name
            file_map[base_file_name] = file_data
                
    return file_map


get_publication_workspace_documents("7ae51ab1-56ed-45db-b5b8-3683bf8724b5")

Initial zip file corrupt, attempting repair...


BadZipFile: EOCD not found in zip file.